In [1]:
%pip install numpy datasets google.generativeai python_dotenv


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cell 1: Fixed Dataset Loading
import numpy as np
from datasets import load_dataset

def load_and_sample_stories():
    dataset = load_dataset('euclaise/writingprompts', split='train')
    indices = np.random.choice(len(dataset), 120, replace=False)
    indices = [int(i) for i in indices]  # Convert numpy.int64 to native int
    return [dataset[i]['story'] for i in indices], indices

stories, story_indices = load_and_sample_stories()


In [6]:
# Cell 2: Sequential API Calls with Rate Limiting and Incremental Saving
import google.generativeai as genai
import os
import re
import time
from dotenv import load_dotenv
from datetime import datetime
import json

load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

def process_story_with_gemini_sequential(story, index):
    model = genai.GenerativeModel('gemini-1.5-flash')
    
    # Call 1: Objective summary
    prompt_summary = f"""Provide an objective summary of the following story. Refer to the narrator as <the narrator>.

Story: {story[:20000]}"""
    response_summary = model.generate_content(prompt_summary)
    summary = response_summary.text.strip()
    print(f"  - Summary completed")
    
    # Call 2: List of characters
    prompt_characters = f"""List all characters in the following story. Include all named entities.

Story: {story[:20000]}"""
    response_characters = model.generate_content(prompt_characters)
    characters = [c.strip() for c in response_characters.text.strip().split('\n') if c.strip()]
    print(f"  - Characters completed")
    
    # Call 3: Short camelCase story ID
    prompt_name = f"""Create a short camelCase name for the following story. This will be used as a story ID.

Story: {story[:20000]}"""
    response_name = model.generate_content(prompt_name)
    name = response_name.text.strip()
    print(f"  - Name completed")
    
    # Call 4: List of environments/settings
    prompt_environment = f"""List all environments and settings in the following story.

Story: {story[:20000]}"""
    response_environment = model.generate_content(prompt_environment)
    environment = [e.strip() for e in response_environment.text.strip().split('\n') if e.strip()]
    print(f"  - Environment completed")
    
    return {
        "storyNum": int(index),
        "name": name,
        "source": story,
        "summary": summary,
        "characters": characters,
        "environment": environment
    }

def save_current_progress(data, filename):
    """Save current progress to file"""
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Progress saved to {filename}")

# Initialize filename for incremental saving
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"extracted_{timestamp}.json"

# Process stories and save results incrementally
extracted_data = []
for idx, (story, story_idx) in enumerate(zip(stories, story_indices)):
    time.sleep(10)  # Rate limiting
    print(f"Processing story {idx+1}/120 (index: {story_idx})")
    try:
        result = process_story_with_gemini_sequential(story, story_idx)
        extracted_data.append(result)
        # Save after each successful processing
        save_current_progress(extracted_data, filename)
        print(f"Completed {idx+1}/120 stories")
    except Exception as e:
        print(f"Error processing story {idx+1} (index: {story_idx}): {str(e)}")
        # Save even if there's an error
        save_current_progress(extracted_data, filename)
        print("Continuing with next story...")
    
    # Additional rate limiting between stories
    if idx < len(stories) - 1:  # Don't sleep after the last story
        print("Waiting 5 seconds before next story...")
        time.sleep(5)

print(f"All processing complete. Final data saved to {filename}")


Processing story 1/120 (index: 76051)
  - Summary completed
  - Characters completed
  - Name completed
  - Environment completed
Progress saved to extracted_20250504_032008.json
Completed 1/120 stories
Waiting 5 seconds before next story...
Processing story 2/120 (index: 55531)
  - Summary completed
  - Characters completed
  - Name completed
  - Environment completed
Progress saved to extracted_20250504_032008.json
Completed 2/120 stories
Waiting 5 seconds before next story...
Processing story 3/120 (index: 186767)
  - Summary completed
  - Characters completed
  - Name completed
  - Environment completed
Progress saved to extracted_20250504_032008.json
Completed 3/120 stories
Waiting 5 seconds before next story...
Processing story 4/120 (index: 75199)
  - Summary completed
  - Characters completed
  - Name completed
  - Environment completed
Progress saved to extracted_20250504_032008.json
Completed 4/120 stories
Waiting 5 seconds before next story...
Processing story 5/120 (index: 

In [1]:
# Cell 4: Inference Functions with Ollama/Mistral Implementation
import csv
import time
import os
import requests
import json

def generate_retelling(extracted_data, output_dir="output"):
    # Format the prompt with data from the extracted story
    characters_str = ", ".join(extracted_data["characters"])
    environments_str = ", ".join(extracted_data["environment"])
    
    prompt_template = f"""Here is a list of objective, third person facts from a story:
    Characters: {characters_str}
    Settings: {environments_str}
    Original story: {extracted_data["summary"]}
    
    Using the information provided, craft a compelling retelling of the story that is from a perspective other than the narrator's. Choose one of the characters or elements in the story to narrate from their point of view.
    
    First, state which character or perspective you've chosen, then write the retelling."""
    
    # Call Ollama API running locally with Mistral
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "mistral:latest",
                "prompt": prompt_template,
                "stream": False
            },
            timeout=120  # 2-minute timeout
        )
        
        # Check if request was successful
        if response.status_code == 200:
            result = response.json()
            retelling = result.get("response", "Error: No response generated")
        else:
            retelling = f"Error: Received status code {response.status_code} from Ollama API"
            print(f"API Error: {response.text}")
    
    except requests.exceptions.RequestException as e:
        retelling = f"Error connecting to Ollama: {str(e)}"
        print(f"Connection error: {str(e)}")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate unique filename with timestamp
    timestamp = time.strftime('%m-%d-%H-%M-%S')
    filename = f"{output_dir}/retell_{extracted_data['name']}_{timestamp}.csv"
    
    # Save the retelling to CSV
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["storyNum", "name", "source", "retelling"])
        writer.writeheader()
        writer.writerow({
            "storyNum": extracted_data["storyNum"],
            "name": extracted_data["name"],
            "source": extracted_data["source"],
            "retelling": retelling
        })
    
    print(f"Retelling saved to {filename}")
    return filename

def batch_generate_retellings(extracted_data_file:str, output_dir="output"):
    """Process all stories from an extracted data file"""
    
    # Load the extracted data
    with open(extracted_data_file, 'r') as f:
        extracted_data_list = json.load(f)
    
    # Process each story
    for idx, story_data in enumerate(extracted_data_list):
        print(f"Generating retelling {idx+1}/{len(extracted_data_list)} for '{story_data['name']}'")
        
        try:
            filename = generate_retelling(story_data, output_dir)
            print(f"Successfully generated retelling: {filename}")
        except Exception as e:
            print(f"Error generating retelling for story {story_data['name']}: {str(e)}")
    
    print(f"All retellings complete. Results saved to {output_dir}/")


In [2]:
batch_generate_retellings("./extracted_20250504_032008.json")

Generating retelling 1/119 for 'JeromeUnirnetEncounter'
Retelling saved to output/retell_JeromeUnirnetEncounter_05-04-04-58-14.csv
Successfully generated retelling: output/retell_JeromeUnirnetEncounter_05-04-04-58-14.csv
Generating retelling 2/119 for 'EarthCommandPrompt'
Retelling saved to output/retell_EarthCommandPrompt_05-04-04-58-49.csv
Successfully generated retelling: output/retell_EarthCommandPrompt_05-04-04-58-49.csv
Generating retelling 3/119 for 'DracolordHelmets'
Retelling saved to output/retell_DracolordHelmets_05-04-04-59-16.csv
Successfully generated retelling: output/retell_DracolordHelmets_05-04-04-59-16.csv
Generating retelling 4/119 for 'twoGunStandoff'
Retelling saved to output/retell_twoGunStandoff_05-04-04-59-45.csv
Successfully generated retelling: output/retell_twoGunStandoff_05-04-04-59-45.csv
Generating retelling 5/119 for 'MomCallsDumpTruck'
Retelling saved to output/retell_MomCallsDumpTruck_05-04-05-00-11.csv
Successfully generated retelling: output/retell_M

In [ ]:
# Cell 5: Evaluation Functions
def evaluate_retelling(original, retelling):
    evaluation_prompt = f"""Evaluate this story retelling using these criteria:
    
    1. Character Selection Valence (1-5):
    2. Character Selection Error (bool):
    3. Factual Consistency (1-5):
    4. Factual Consistency Error (bool):
    5. Character Consistency (1-5):
    
    Original: {original}
    Retelling: {retelling}"""
    
    # Implementation for Llama-8B evaluation would go here
    # Pseudocode for demonstration:
    return {
        "valence": 4,
        "selection_error": False,
        "consistency": 3,
        "fact_error": False,
        "character_consistency": 4,
        "reasoning": {
            "valence": "Chose secondary character mentioned in paragraph 3",
            "consistency": "Maintained plot but added new internal monologue",
            "character": "Consistent voice but some anachronisms"
        }
    }

def batch_evaluate(output_dir):
    # Implementation for processing all retellings
    pass
